In [1]:
import numpy as np
# Patch for NumPy >= 1.24 where np.bool8 was removed
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_
if not hasattr(np, "np.float_"):
    np.float_ = np.float64
if not hasattr(np, "np.complex_"):
    np.complex_ = np.complex128
if not hasattr(np, "np.unicode_"):
    np.unicode_ = np.str_

# import torch
# # Quick patch for PyTorch 2.6+ to fix UnpicklingError
# orig_torch_load = torch.load
# torch.load = lambda f, *args, **kwargs: orig_torch_load(f, *args, weights_only=False, **kwargs)
# # def safe_torch_load(*args, **kwargs):
# #     # Ensure weights_only is always present but never duplicated
# #     kwargs.setdefault("weights_only", False)
# #     return orig_torch_load(*args, **kwargs)
# # torch.load = safe_torch_load


import logging
def recbole_reset_logs():
    logging.shutdown()
    for handler in logging.root.handlers[:]:
        handler.close()
        logging.root.removeHandler(handler)

C:\Users\martin\AppData\Local\Temp\ipykernel_13076\3355334233.py:3: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not hasattr(np, "bool8"):


In [2]:
from recbole.quick_start import run_recbole

MovieLens, Jester, Amazon, Netflix = range(4)
ItemKNN, BPR, NeuMF, GCMC = range(4)
expR, impR, impB = range(3)

Model = GCMC                     # <--- Make changes here
Dataset = Jester                  # <--- Make changes here                      
Datatype = [expR, impR, impB]                     # <--- Make changes here

for t in Datatype:
    models = ['ItemKNN', 'BPR', 'NeuMF', 'GCMC']
    datasets = ['ml-100k', 'jester', 'amazon_all_beauty', 'netflix']
    datatypes = ['expR', 'impR', 'impB']
    model = models[Model]            
    dataset = datasets[Dataset]       
    datatype = datatypes[t]#datatypes[Datatype]

    config_dict = {
        # --- Model + data selection ---
        'model': model,
        'data_path': f'./data/split/{dataset}',                                
        'dataset': f'{dataset}--{datatype}',      
        'benchmark_filename': ['train','valid','test'],                    

        # 'save_dataset': True,
        # 'dataset_save_path': f'./data/saved/{dataset}--{datatype}',
        
        # Disable RecBole auto-splitting
        'eval_args': {
            'split': {'RS': [0, 0, 0]},     # no random split
            'mode': 'full'
            #'mode': {'valid': 'uni200', 'test': 'uni200'}
            #'mode': {'valid': 'train', 'test': 'train'}
        },
        'test_args': {
            'mode': 'full' # Full ranking (no sampling)
            #'mode': {'valid': 'uni200', 'test': 'uni200'}
            #'mode': {'valid': 'train', 'test': 'train'}
        }, 

        # --- Evaluation targets ---
        'topk': [10, 20],   # Top-k recommendations to consider
        'metrics': ['Precision', 'Recall', 'MRR', 'Hit', 'NDCG'],
        #'metrics': ['RMSE', 'MAE'],

        # --- Training schedule + early stopping ---
        'epochs': 50,                   # Max epochs
        'eval_step': 1,                 # Evaluate every epoch
        'stopping_step': 5,             # Stop if no improvement for 5 evals (i.e., 5 epochs)
        'valid_metric': 'NDCG@10',      # Choose ONE metric for early stopping
        'valid_metric_bigger': True     # because higher NDCG is better
    }

    recbole_reset_logs()    
    run_recbole(config_dict=config_dict)
    recbole_reset_logs()

c:\Users\martin\miniconda3\envs\bp\Lib\site-packages\ray\_private\parameter.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
20 Nov 13:35    INFO  ['c:\\Users\\martin\\miniconda3\\envs\\bp\\Lib\\site-packages\\ipykernel_launcher.py', '--f=c:\\Users\\martin\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3d3965cc7e088cddf2919524a152e792a8fcad876.json']
20 Nov 13:35    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = ./data/split/jester\jester--expR
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 50
train_batch_size = 2048
learner = adam
learning_rate = 0.

KeyboardInterrupt: 